In [1]:
import itertools
import numpy as np
import cvxpy as cp
from itertools import product

In [2]:
def minPIC_multicarrier_solver(U, H, min_rate, F):
    """
    Solves the minPIC optimization problem for multi-carrier systems.
    
    :param U: Number of users (and receivers)
    :param H: Channel matrix of size (F, U, U) - channel gains for each frequency
    :param min_rate: Minimum required total data rates for each user (array of size U)
    :param F: Number of frequency subcarriers
    :return: (best_sol, best_powers, best_data_rates, best_pi)
    """
    
    LN2 = np.log(2.0)
    
    def cap_scalar(H_vec, r_list, sigma2=1.0):
        """
        Returns (1/2) * log2(1 + (sum_k (H_k^2 * r_k))/sigma2) as a CVXPY expression.
        H_vec: 1D numpy array of constants (length K)
        r_list: list of CVXPY scalar variables (length K)
        """
        if len(H_vec) == 0:
            return cp.Constant(0.0)
        h2 = np.square(H_vec)  # constant weights
        r = cp.hstack(list(r_list))  # (K,)
        return 0.5 * cp.log1p((h2 @ r) / sigma2) / LN2
    
    permutations = list(itertools.permutations(range(U))) # TODO: bring this line back and comment the next line
    # permutations = [tuple(range(U))] 
    
    #-----------------------------------------------------------------------
    # Helper function: solve once for a given imp_factor
    #-----------------------------------------------------------------------
    def solve_for_imp_factor(imp_factor):
        best_sol_local = float('inf')
        best_powers_local = None
        best_data_rates_local = None
        constraints_tight_global = False
        
        for pi in permutations:
            # Define optimization variables - now indexed by frequency too
            Rxx = {(i, j, f): cp.Variable(nonneg=True) 
                   for i in range(U) for j in range(U) for f in range(F)}
            b   = {(i, j, f): cp.Variable() 
                   for i in range(U) for j in range(U) for f in range(F)}
            c   = {(i, f): cp.Variable() 
                   for i in range(U) for f in range(F)}
            
            constraints = []
            
            # Process each receiver and each frequency
            for i in range(U):
                # S1, S2, S3 classification (same as single-tone)
                S1 = [(i, j) for j in range(U)]
                S2 = [(j, i) for j in range(U) if j != i]
                S3 = [(j, k) for j in range(U) for k in range(U) 
                      if (j, k) not in (S1 + S2)]
                
                # Decoding order based on pi
                ordered_S1_S2 = sorted(
                    S1 + S2,
                    key=lambda x: (pi.index(x[0]), pi.index(x[1]))
                )
                decoding_order = list(ordered_S1_S2)
                
                # Now loop over each frequency
                for f in range(F):
                    # c[i,f] <= capacity from unimportant signals at freq f
                    if len(S3) > 0:
                        H_unimportant_f_vec = np.array([H[f, i, j] for (j, k) in S3])
                        Rxx_unimportant_list = [Rxx[j, k, f] for (j, k) in S3]
                        constraints.append(
                            c[i, f] <= cap_scalar(H_unimportant_f_vec, Rxx_unimportant_list)
                        )
                    else:
                        # If S3 is empty, c[i,f] can be at most 0
                        constraints.append(c[i, f] == 0)
                    
                    # c[i,f] >= 0
                    constraints.append(c[i, f] >= 0)
                    
                    # Decoding-order constraints for frequency f
                    cumulative_sum = c[i, f]
                    
                    for idx, (j, k) in enumerate(decoding_order):
                        curr_pairs = S3 + decoding_order[:idx+1]
                        H_eff_vec = np.array([H[f, i, jj] for (jj, kk) in curr_pairs])
                        R_eff_list = [Rxx[jj, kk, f] for (jj, kk) in curr_pairs]
                        constraints.append(
                            cumulative_sum + b[j, k, f] <= cap_scalar(H_eff_vec, R_eff_list)
                        )
                        cumulative_sum += b[j, k, f]
                    
                    # b[i,j,f] >= 0 for all j and f
                    for j in range(U):
                        constraints.append(b[i, j, f] >= 0)
                
                # Rate constraint: sum over all j and all frequencies f
                constraints.append(
                    sum(b[i, j, f] for j in range(U) for f in range(F)) >= min_rate[i]
                )
            
            # Objective: minimize total power across all frequencies minus weighted interference
            objective = cp.Minimize(
                cp.sum(cp.hstack([Rxx[i, j, f] for i in range(U) for j in range(U) for f in range(F)])) -
                imp_factor * cp.sum(cp.hstack([c[i, f] for i in range(U) for f in range(F)]))
            )
            
            # Solve
            prob = cp.Problem(objective, constraints)
            try:
                prob.solve(solver=cp.SCS, max_iters=200000, warm_start=True, verbose=False)
            except cp.SolverError:
                continue  # skip if solver fails
            
            # --- Diagnostic: log feasibility status ---
            if prob.status not in [cp.OPTIMAL, cp.OPTIMAL_INACCURATE]:
                print(f"[Diagnostic] imp_factor={imp_factor}, π={pi} -> Solver status = {prob.status}")
            else:
                # solver found a feasible point, but maybe not tight
                if np.isfinite(prob.value):
                    print(f"[Diagnostic] imp_factor={imp_factor}, π={pi} feasible, obj={prob.value:.4f} (checking tightness next...)")

            print ("STATUS is ", prob.status)
            print ("values are ", prob.value, best_sol_local)
            if prob.status in [cp.OPTIMAL, cp.OPTIMAL_INACCURATE] and prob.value < best_sol_local:
                # Check if all c[i,f] constraints are tight
                print ("HUNGAMA")
                all_tight = True
                for i in range(U):
                    S1 = [(i, j) for j in range(U)]
                    S2 = [(j, i) for j in range(U) if j != i]
                    S3 = [(j, k) for j in range(U) for k in range(U)
                          if (j, k) not in (S1 + S2)]
                    
                    for f in range(F):
                        if len(S3) > 0:
                            H_unimportant_f_vec = np.array([H[f, i, j] for (j, k) in S3])
                            Rxx_unimportant_val = np.array([Rxx[j, k, f].value for (j, k) in S3])
                            
                            h2 = np.square(H_unimportant_f_vec)
                            rhs_val = 0.5 * np.log1p(np.sum(h2 * Rxx_unimportant_val)) / LN2
                            
                            lhs_val = c[i, f].value
                            
                            # Guard against None or invalid values
                            if lhs_val is None or not np.isfinite(lhs_val):
                                all_tight = False
                                break
                            
                            # If difference bigger than threshold => not tight
                            if abs(lhs_val - rhs_val) > 1e-1:
                                print ("Breaking out because of tightness constraint ", lhs_val, rhs_val)
                                all_tight = False
                                break
                    
                    if not all_tight:
                        print ("MOTAGAMA")
                        break
                
                if all_tight:
                    # Update best local solution
                    best_sol_local = prob.value
                    best_powers_local = {(i, j, f): Rxx[i, j, f].value 
                                        for i in range(U) for j in range(U) for f in range(F)}
                    best_data_rates_local = {i: np.sum(b[i, j, f].value for j in range(U) for f in range(F)) for i in range(U)}
                    constraints_tight_global = True
        
        return best_sol_local, best_powers_local, best_data_rates_local, pi, constraints_tight_global
    
    #-----------------------------------------------------------------------
    # 1) BRACKETING: find some highFactor where constraints become tight
    #-----------------------------------------------------------------------
    lowFactor = 1.0 # TODO: if automatically satisfied at lowFactor, keep binary searching below
    highFactor = lowFactor
    maxFactor = 1e13
    found_tight = False
    best_pi = None
    
    while highFactor <= maxFactor:
        best_sol, best_powers, best_data_rates, best_pi_try, c_tight = solve_for_imp_factor(highFactor)
        if c_tight:
            found_tight = True
            best_pi = best_pi_try            # Seed with first tight pi
            break
        else:
            lowFactor = highFactor
            highFactor *= 2
    
    if not found_tight:
        print("No tight solution found up to imp_factor =", highFactor)
        return None, None, None, None
    
    #-----------------------------------------------------------------------
    # 2) BINARY SEARCH: find the minimal factor in [lowFactor, highFactor]
    #-----------------------------------------------------------------------
    best_sol_global = best_sol
    best_powers_global = best_powers
    best_data_rates_global = best_data_rates
    best_pi_global = best_pi                 # Track global best pi
    
    for _ in range(20):
        if (highFactor - lowFactor) < 1e-2:
            break
        midFactor = 0.5 * (lowFactor + highFactor)
        
        best_sol_mid, best_powers_mid, best_data_rates_mid, best_pi_mid, c_tight_mid = solve_for_imp_factor(midFactor)
        
        if c_tight_mid:
            highFactor = midFactor
            if best_sol_mid < best_sol_global:
                best_sol_global = best_sol_mid
                best_powers_global = best_powers_mid
                best_data_rates_global = best_data_rates_mid
        else:
            lowFactor = midFactor
    
    print("Binary search done. Found factor in [", lowFactor, ",", highFactor, "].")
    print("Best objective value:", best_sol_global)
    # # TODO: uncomment above, comment below
    # # -----------------------------------------------------------------------
    # # Simplified single imp_factor run (skip binary search)
    # # -----------------------------------------------------------------------
    # imp_factor = 100.0
    # best_sol_global, best_powers_global, best_data_rates_global, _ = solve_for_imp_factor(imp_factor)

    # print(f"Fixed imp_factor = {imp_factor}")
    # print("Best objective value:", best_sol_global)
    
    return best_sol_global, best_powers_global, best_data_rates_global, best_pi_global

In [4]:
LN2 = np.log(2.0)

def _cap_scalar(H_vec, r_list, sigma2=1.0):
    """(1/2) * log2(1 + sum_k |H_k|^2 * r_k / sigma2) with CVXPY values as floats."""
    if len(H_vec) == 0:
        return 0.0
    h2 = np.square(np.asarray(H_vec, dtype=float))  # (K,)
    r  = np.array([float(x) for x in r_list], dtype=float)  # (K,)
    x  = (h2 * r).sum() / float(sigma2)
    return 0.5 * np.log1p(x) / LN2

def verify_with_order(U, F, H, opt_powers, min_rate, pi, noise_power=1.0, tol=1e-9):
    """
    Recompute achievable user rates respecting the SAME decoding order used in the solver.

    Parameters
    ----------
    U, F : ints
    H    : (F, U, U) array, H[f,i,j] = gain TX j -> RX i at tone f
    opt_powers : dict {(i,j,f): Rxx_value}  (from your solver)
    min_rate   : array-like of length U (TOTAL per user across all tones)
    pi         : tuple/list of length U, global decoding order over transmitters (e.g., (0,1,2,3))
    noise_power: scalar (default 1.0)
    tol        : small clip for numerical negatives

    Returns
    -------
    achievable_user_rate : dict {i: achievable upper bound under this order}
    per_tone_user_rate   : array (U,F) with the per-tone contributions to each user
    """
    # sanity helpers to fetch Rxx[(j,k,f)] safely (0.0 if missing)
    def P(j,k,f):
        return float(opt_powers.get((j,k,f), 0.0))

    per_tone_user_rate = np.zeros((U, F), dtype=float)

    for i in range(U):  # receiver i
        # Sets per your notebook
        S1 = [(i, j) for j in range(U)]
        S2 = [(j, i) for j in range(U) if j != i]
        S3 = [(j, k) for j in range(U) for k in range(U) if (j, k) not in (S1 + S2)]

        # Decoding order based on global transmitter order pi
        ordered_S1_S2 = sorted(S1 + S2, key=lambda x: (pi.index(x[0]), x[1]))
        chain = list(ordered_S1_S2)  # sequence of pairs (j,k)

        for f in range(F):
            # Step 0: unimportant pool capacity (S3 only)
            H_S3   = [H[f, i, j] for (j,k) in S3]
            R_S3   = [P(j,k,f)   for (j,k) in S3]
            C_prev = _cap_scalar(H_S3, R_S3, sigma2=noise_power)

            # Walk the chain; accumulate only increments for pairs in S1 (i.e., first index == i)
            user_i_sum_f = 0.0
            seen_pairs   = list(S3)  # what is decoded so far at RX i

            for (j,k) in chain:
                seen_pairs.append((j,k))
                H_seen = [H[f, i, jj] for (jj,kk) in seen_pairs]
                R_seen = [P(jj,kk,f) for (jj,kk) in seen_pairs]
                C_curr = _cap_scalar(H_seen, R_seen, sigma2=noise_power)
                delta  = max(0.0, C_curr - C_prev)  # clip small negatives
                C_prev = C_curr

                # Only S1 increments contribute to user i's *own* rate (consistent with your rate constraint)
                if j == i:
                    user_i_sum_f += delta

            per_tone_user_rate[i, f] = user_i_sum_f

    achievable_user_rate = {i: float(per_tone_user_rate[i,:].sum()) for i in range(U)}

    print("\n=== Order-aware Verification (same SIC order) ===")
    print(f"Decoding order (TX permutation): {pi}")
    for i in range(U):
        print(f"User {i}:  achievable={achievable_user_rate[i]:.9f}   "
              f"(target={float(min_rate[i]):.9f})   "
              f"margin={achievable_user_rate[i]-float(min_rate[i]):.9e}")
    return achievable_user_rate, per_tone_user_rate

## Test Case 1: 2 Users, 4 Subcarriers

In [4]:
# Test with 2 users and 4 frequency subcarriers
U = 2
F = 4

# Create frequency-selective channel
# H[f, i, j] represents channel from transmitter j to receiver i at frequency f
H = np.zeros((F, U, U))

# Set up different channel gains for different frequencies
for f in range(F):
    H[f] = np.eye(U)
    # Add frequency-selective cross-channel interference
    H[f, 0, 1] = 0.3 + 0.1 * f / F  # Varies with frequency
    H[f, 1, 0] = 0.2 + 0.15 * f / F

min_rate = np.array([0.5, 0.5])

opt_sol, opt_powers, opt_data_rates, best_pi = minPIC_multicarrier_solver(U, H, min_rate, F)

print("Optimal decoding order ", best_pi)

print("\n=== Results ===")
print("Optimal Data Rates:", opt_data_rates)
print("Optimal objective value:", opt_sol)

# Calculate total power
if opt_powers is not None:
    total_power = sum(opt_powers.values())
    print(f"Total Power: {total_power}")
    
    # Show power allocation per frequency
    print("\nPower per frequency:")
    for f in range(F):
        power_f = sum(opt_powers[i, j, f] for i in range(U) for j in range(U))
        print(f"  Frequency {f}: {power_f:.4f}")
    
    # Detailed per-(i,j,f) power distribution
    print("\n=== Detailed Power Allocation per Subcarrier and Link ===")
    for f in range(F):
        print(f"\nSubcarrier {f}:")
        for i in range(U):
            for j in range(U):
                p = opt_powers[(i, j, f)]
                print(f"  Rxx[{i},{j},{f}] = {p:.6f}")
    
    # Aggregated views
    print("\n=== Aggregate Power per Transmitter (summed over receivers & freqs) ===")
    for j in range(U):
        total_tx = sum(opt_powers[i, j, f] for i in range(U) for f in range(F))
        print(f"  TX {j}: {total_tx:.6f}")
    
    print("\n=== Aggregate Power per Receiver (summed over transmitters & freqs) ===")
    for i in range(U):
        total_rx = sum(opt_powers[i, j, f] for j in range(U) for f in range(F))
        print(f"  RX {i}: {total_rx:.6f}")
        
        
achieved_rates, rate_matrix = verify_with_order(U, F, H, opt_powers, min_rate, best_pi)

[Diagnostic] imp_factor=10.0, π=(0, 1) feasible, obj=0.3766 (checking tightness next...)
[Diagnostic] imp_factor=10.0, π=(1, 0) feasible, obj=0.3765 (checking tightness next...)
Binary search done. Found factor in [ 10.0 , 10.0 ].
Best objective value: 0.37652306151540227
Optimal decoding order  (1, 0)

=== Results ===
Optimal Data Rates: {0: 0.5000000011623686, 1: 0.5000000012426269}
Optimal objective value: 0.37652306151540227
Total Power: 1.8202652543506872

Power per frequency:
  Frequency 0: 0.0000
  Frequency 1: 0.0095
  Frequency 2: 0.4623
  Frequency 3: 1.3484

=== Detailed Power Allocation per Subcarrier and Link ===

Subcarrier 0:
  Rxx[0,0,0] = 0.000000
  Rxx[0,1,0] = 0.000000
  Rxx[1,0,0] = 0.000001
  Rxx[1,1,0] = 0.000001

Subcarrier 1:
  Rxx[0,0,1] = 0.009538
  Rxx[0,1,1] = 0.000000
  Rxx[1,0,1] = 0.000002
  Rxx[1,1,1] = 0.000002

Subcarrier 2:
  Rxx[0,0,2] = 0.256944
  Rxx[0,1,2] = 0.000000
  Rxx[1,0,2] = 0.000000
  Rxx[1,1,2] = 0.205399

Subcarrier 3:
  Rxx[0,0,3] = 0.6

## Test Case 2: 3 Users, 8 Subcarriers

In [43]:
# Test with 3 users and 8 frequency subcarriers
U = 3
F = 1

# Create frequency-selective channel
H = np.zeros((F, U, U))

for f in range(F):
    H[f] = np.eye(U)
    # Add frequency-selective interference pattern
    H[f, 0, 1] = 0.5 * np.cos(2 * np.pi * f / F)
    H[f, 0, 2] = 0.001
    H[f, 1, 0] = 0.001
    H[f, 1, 2] = 0.3 * np.sin(2 * np.pi * f / F)
    H[f, 2, 0] = 0.001
    H[f, 2, 1] = 0.001

min_rate = np.array([0.5, 0.5, 0.5])

opt_sol, opt_powers, opt_data_rates, best_pi = minPIC_multicarrier_solver(U, H, min_rate, F)

print("Optimal decoding order ", best_pi)

print("\n=== Results ===")
print("Optimal Data Rates:", opt_data_rates)
print("Optimal objective value:", opt_sol)

if opt_powers is not None:
    total_power = sum(opt_powers.values())
    print(f"Total Power: {total_power}")
    
    print("\nPower per frequency:")
    for f in range(F):
        power_f = sum(opt_powers[i, j, f] for i in range(U) for j in range(U))
        print(f"  Frequency {f}: {power_f:.4f}")
    
    # Detailed per-(i,j,f) power distribution
    print("\n=== Detailed Power Allocation per Subcarrier and Link ===")
    for f in range(F):
        print(f"\nSubcarrier {f}:")
        for i in range(U):
            for j in range(U):
                p = opt_powers[(i, j, f)]
                print(f"  Rxx[{i},{j},{f}] = {p:.6f}")
    
    # Aggregated views
    print("\n=== Aggregate Power per Transmitter (summed over receivers & freqs) ===")
    for j in range(U):
        total_tx = sum(opt_powers[i, j, f] for i in range(U) for f in range(F))
        print(f"  TX {j}: {total_tx:.6f}")
    
    print("\n=== Aggregate Power per Receiver (summed over transmitters & freqs) ===")
    for i in range(U):
        total_rx = sum(opt_powers[i, j, f] for j in range(U) for f in range(F))
        print(f"  RX {i}: {total_rx:.6f}")
        
        
achieved_rates, rate_matrix = verify_with_order(U, F, H, opt_powers, min_rate, best_pi)

Decoding order  (0, 1, 2)  at receiver  0
[(2, 0), (1, 0), (0, 2), (0, 1), (0, 0)]
Decoding order  (0, 1, 2)  at receiver  1
[(2, 1), (1, 2), (1, 1), (1, 0), (0, 1)]
Decoding order  (0, 1, 2)  at receiver  2
[(2, 2), (2, 1), (2, 0), (1, 2), (0, 2)]
[Diagnostic] imp_factor=1.0, π=(0, 1, 2) feasible, obj=2.7500 (checking tightness next...)
Breaking out because of tightness constraint  3.186913070652022e-07 0.16096437900200586
Decoding order  (0, 2, 1)  at receiver  0
[(1, 0), (2, 0), (0, 1), (0, 2), (0, 0)]
Decoding order  (0, 2, 1)  at receiver  1
[(1, 1), (1, 2), (1, 0), (2, 1), (0, 1)]
Decoding order  (0, 2, 1)  at receiver  2
[(1, 2), (2, 1), (2, 2), (2, 0), (0, 2)]
[Diagnostic] imp_factor=1.0, π=(0, 2, 1) feasible, obj=2.7500 (checking tightness next...)
Breaking out because of tightness constraint  1.2792090872790758e-05 0.16096134208255225
Decoding order  (1, 0, 2)  at receiver  0
[(2, 0), (0, 2), (0, 0), (0, 1), (1, 0)]
Decoding order  (1, 0, 2)  at receiver  1
[(2, 1), (0, 1), (

## Test Case 3: Comparison with Single-Tone (F=1)

In [39]:
# Verify that F=1 case matches single-tone behavior
U = 2
F = 1

H = np.zeros((F, U, U))
H[0] = np.eye(U)
H[0, 0, 1] = 0.4
H[0, 1, 0] = 0.6

min_rate = np.array([0.5, 0.5])

opt_sol, opt_powers, opt_data_rates, best_pi = minPIC_multicarrier_solver(U, H, min_rate, F)

print("Optimal decoding order ", best_pi)

print("\n=== Single Frequency Results (F=1) ===")
print("Optimal Data Rates:", opt_data_rates)
print("Optimal objective value:", opt_sol)

if opt_powers is not None:
    total_power = sum(opt_powers.values())
    print(f"Total Power: {total_power}")
    print("\nPower allocation:")
    for (i, j, f), p in opt_powers.items():
        if p > 1e-6:
            print(f"  Rxx[{i},{j},{f}] = {p:.6f}")
            
achieved_rates, rate_matrix = verify_with_order(U, F, H, opt_powers, min_rate, best_pi)

Binary search done. Found factor in [ 3.4404296875 , 3.44140625 ].
Best objective value: 1.0162054278445194
Optimal decoding order  (1, 0)

=== Single Frequency Results (F=1) ===
Optimal Data Rates: {0: 0.49999999619865215, 1: 0.4999999975950564}
Optimal objective value: 1.0162054278445194
Total Power: 2.674015794581969

Power allocation:
  Rxx[0,0,0] = 1.230896
  Rxx[1,1,0] = 1.443120

=== Order-aware Verification (same SIC order) ===
Decoding order (TX permutation): (1, 0)
User 0:  achievable=0.499999108   (target=0.500000000)   margin=-8.922718536e-07
User 1:  achievable=0.499999273   (target=0.500000000)   margin=-7.269010154e-07


## Analysis: Power Allocation Across Frequencies

## Test Case 5: 4 Users, 8 Subcarriers - High Interference Case

In [38]:
# This is a challenging case with strong cross-channel interference
# Converted from single-carrier scenario to multi-carrier

U = 4
F = 64  # Multiple subcarriers for frequency diversity

# Create frequency-selective channel
# Base structure from the single-carrier case
H = np.zeros((F, U, U))

for f in range(F):
    # Start with identity (direct channels)
    H[f] = np.eye(U)
    
    # Add strong interference from user 0 to receivers 1, 2, 3
    H[f, 0, 1] = 0.9 * (1 + 0.1 * np.sin(2 * np.pi * f / F))  # frequency-selective
    H[f, 0, 2] = 0.9 * (1 + 0.1 * np.cos(2 * np.pi * f / F))
    H[f, 0, 3] = 1.0
    
    # Weak cross-interference for other links
    H[f, 1, 0] = 0.001
    H[f, 1, 2] = 0.001

min_rate = np.array([0.5, 0.5, 0.5, 0.5])

opt_sol, opt_powers, opt_data_rates, best_pi = minPIC_multicarrier_solver(U, H, min_rate, F)

print("Optimal decoding order ", best_pi)

print("\n=== 4-User High Interference Case (F=8) ===")
print("Optimal Data Rates:", opt_data_rates)
print("Optimal objective value:", opt_sol)

if opt_powers is not None:
    total_power = sum(opt_powers.values())
    print(f"\nTotal Power: {total_power}")
    
    print("\nPower per frequency:")
    for f in range(F):
        power_f = sum(opt_powers[i, j, f] for i in range(U) for j in range(U))
        print(f"  f={f:2d}: {power_f:.6f}")
    
    print("\nPower allocation per user per frequency:")
    for i in range(U):
        print(f"\nUser {i}:")
        for f in range(F):
            power_i_f = sum(opt_powers[i, j, f] for j in range(U))
            if power_i_f > 1e-6:
                print(f"  f={f:2d}: {power_i_f:.6f}")
    
    # Detailed per-(i,j,f) power distribution
    print("\n=== Detailed Power Allocation per Subcarrier and Link ===")
    for f in range(F):
        print(f"\nSubcarrier {f}:")
        for i in range(U):
            for j in range(U):
                p = opt_powers[(i, j, f)]
                if p > 1e-6:  # Only show non-zero powers
                    print(f"  Rxx[{i},{j},{f}] = {p:.6f}")
    
    # Aggregated views
    print("\n=== Aggregate Power per Transmitter (summed over receivers & freqs) ===")
    for j in range(U):
        total_tx = sum(opt_powers[i, j, f] for i in range(U) for f in range(F))
        print(f"  TX {j}: {total_tx:.6f}")
    
    print("\n=== Aggregate Power per Receiver (summed over transmitters & freqs) ===")
    for i in range(U):
        total_rx = sum(opt_powers[i, j, f] for j in range(U) for f in range(F))
        print(f"  RX {i}: {total_rx:.6f}")
        
achieved_rates, rate_matrix = verify_with_order(U, F, H, opt_powers, min_rate, best_pi)

Decoding order  (0, 1, 2, 3)
[(3, 0), (2, 0), (1, 0), (0, 3), (0, 2), (0, 1), (0, 0)]
Decoding order  (0, 1, 2, 3)
[(3, 1), (2, 1), (1, 3), (1, 2), (1, 1), (1, 0), (0, 1)]
Decoding order  (0, 1, 2, 3)
[(3, 2), (2, 3), (2, 2), (2, 1), (2, 0), (1, 2), (0, 2)]
Decoding order  (0, 1, 2, 3)
[(3, 3), (3, 2), (3, 1), (3, 0), (2, 3), (1, 3), (0, 3)]
[Diagnostic] imp_factor=1.0, π=(0, 1, 2, 3) feasible, obj=1.1523 (checking tightness next...)
Decoding order  (0, 1, 3, 2)
[(2, 0), (3, 0), (1, 0), (0, 3), (0, 2), (0, 1), (0, 0)]
Decoding order  (0, 1, 3, 2)
[(2, 1), (3, 1), (1, 3), (1, 2), (1, 1), (1, 0), (0, 1)]
Decoding order  (0, 1, 3, 2)
[(2, 3), (2, 2), (2, 1), (2, 0), (3, 2), (1, 2), (0, 2)]
Decoding order  (0, 1, 3, 2)
[(2, 3), (3, 3), (3, 2), (3, 1), (3, 0), (1, 3), (0, 3)]


KeyboardInterrupt: 

In [6]:
# Detailed analysis with visualization-ready data
U = 2
F = 16

H = np.zeros((F, U, U))

# Create realistic frequency-selective channel
for f in range(F):
    H[f] = np.eye(U)
    # Rayleigh-like fading pattern
    H[f, 0, 1] = 0.5 * (1 + 0.3 * np.cos(4 * np.pi * f / F))
    H[f, 1, 0] = 0.3 * (1 + 0.5 * np.sin(2 * np.pi * f / F))

min_rate = np.array([1.0, 1.0])

opt_sol, opt_powers, opt_data_rates, best_pi = minPIC_multicarrier_solver(U, H, min_rate, F)

print("\n=== Multi-Carrier Analysis (F=16) ===")
print("Optimal Data Rates:", opt_data_rates)
print("Optimal objective value:", opt_sol)

if opt_powers is not None:
    total_power = sum(opt_powers.values())
    print(f"\nTotal Power: {total_power}")
    
    print("\nPower allocation per frequency:")
    for f in range(F):
        power_f = sum(opt_powers[i, j, f] for i in range(U) for j in range(U))
        print(f"  f={f:2d}: {power_f:.6f}")
    
    print("\nPower allocation per user per frequency:")
    for i in range(U):
        print(f"\nUser {i}:")
        for f in range(F):
            power_i_f = sum(opt_powers[i, j, f] for j in range(U))
            if power_i_f > 1e-6:
                print(f"  f={f:2d}: {power_i_f:.6f}")
    
    # Detailed per-(i,j,f) power distribution
    print("\n=== Detailed Power Allocation per Subcarrier and Link ===")
    for f in range(F):
        print(f"\nSubcarrier {f}:")
        for i in range(U):
            for j in range(U):
                p = opt_powers[(i, j, f)]
                if p > 1e-6:  # Only show non-zero powers
                    print(f"  Rxx[{i},{j},{f}] = {p:.6f}")
    
    # Aggregated views
    print("\n=== Aggregate Power per Transmitter (summed over receivers & freqs) ===")
    for j in range(U):
        total_tx = sum(opt_powers[i, j, f] for i in range(U) for f in range(F))
        print(f"  TX {j}: {total_tx:.6f}")
    
    print("\n=== Aggregate Power per Receiver (summed over transmitters & freqs) ===")
    for i in range(U):
        total_rx = sum(opt_powers[i, j, f] for j in range(U) for f in range(F))
        print(f"  RX {i}: {total_rx:.6f}")
        
achieved_rates, rate_matrix = verify_with_order(U, F, H, opt_powers, min_rate, best_pi)

[Diagnostic] imp_factor=10.0, π=(0, 1) feasible, obj=-22.2318 (checking tightness next...)
[Diagnostic] imp_factor=10.0, π=(1, 0) feasible, obj=-22.2320 (checking tightness next...)
Binary search done. Found factor in [ 10.0 , 10.0 ].
Best objective value: -22.23199147863366

=== Multi-Carrier Analysis (F=16) ===
Optimal Data Rates: {0: 0.999999999918611, 1: 1.0000000000056046}
Optimal objective value: -22.23199147863366

Total Power: 52.194062245205984

Power allocation per frequency:
  f= 0: 4.847184
  f= 1: 4.491570
  f= 2: 4.362326
  f= 3: 2.784281
  f= 4: 2.275221
  f= 5: 2.784281
  f= 6: 4.362326
  f= 7: 4.491570
  f= 8: 4.847184
  f= 9: 4.491037
  f=10: 3.213495
  f=11: 0.769526
  f=12: 0.000002
  f=13: 0.769526
  f=14: 3.213495
  f=15: 4.491037

Power allocation per user per frequency:

User 0:
  f= 0: 0.000633
  f= 1: 0.000414
  f= 2: 1.148836
  f= 3: 2.014754
  f= 4: 2.275221
  f= 5: 2.014754
  f= 6: 1.148836
  f= 7: 0.000414
  f= 8: 0.000633
  f=10: 0.000018
  f=12: 0.000002

In [16]:
## Basic multi tone working
# Detailed analysis with visualization-ready data
U = 2
F = 2

H = np.zeros((F, U, U))

# Create realistic frequency-selective channel
for f in range(F):
    # H[f] = np.eye(U)
    H[f] = np.ones((U,U))
    # # Rayleigh-like fading pattern
    # H[f, 0, 1] = 0.5 * (1 + 0.3 * np.cos(4 * np.pi * f / F))
    # H[f, 1, 0] = 0.3 * (1 + 0.5 * np.sin(2 * np.pi * f / F))

# min_rate = np.array([1.0, 1.0])
min_rate = np.array([0.55, 0.55])

opt_sol, opt_powers, opt_data_rates, best_pi = minPIC_multicarrier_solver(U, H, min_rate, F)

print("\n=== Multi-Carrier Analysis (F=16) ===")
print("Optimal Data Rates:", opt_data_rates)
print("Optimal objective value:", opt_sol)

if opt_powers is not None:
    total_power = sum(opt_powers.values())
    print(f"\nTotal Power: {total_power}")
    
    print("\nPower allocation per frequency:")
    for f in range(F):
        power_f = sum(opt_powers[i, j, f] for i in range(U) for j in range(U))
        print(f"  f={f:2d}: {power_f:.6f}")
    
    print("\nPower allocation per user per frequency:")
    for i in range(U):
        print(f"\nUser {i}:")
        for f in range(F):
            power_i_f = sum(opt_powers[i, j, f] for j in range(U))
            if power_i_f > 1e-6:
                print(f"  f={f:2d}: {power_i_f:.6f}")
    
    # Detailed per-(i,j,f) power distribution
    print("\n=== Detailed Power Allocation per Subcarrier and Link ===")
    for f in range(F):
        print(f"\nSubcarrier {f}:")
        for i in range(U):
            for j in range(U):
                p = opt_powers[(i, j, f)]
                if p > 1e-6:  # Only show non-zero powers
                    print(f"  Rxx[{i},{j},{f}] = {p:.6f}")
    
    # Aggregated views
    print("\n=== Aggregate Power per Transmitter (summed over receivers & freqs) ===")
    for j in range(U):
        total_tx = sum(opt_powers[i, j, f] for i in range(U) for f in range(F))
        print(f"  TX {j}: {total_tx:.6f}")
    
    print("\n=== Aggregate Power per Receiver (summed over transmitters & freqs) ===")
    for i in range(U):
        total_rx = sum(opt_powers[i, j, f] for j in range(U) for f in range(F))
        print(f"  RX {i}: {total_rx:.6f}")
        
achieved_rates, rate_matrix = verify_with_order(U, F, H, opt_powers, min_rate, best_pi)

[Diagnostic] imp_factor=1.0, π=(0, 1) feasible, obj=0.9282 (checking tightness next...)
Breaking out because of tightness constraint  1.585328531157732e-05 0.07548697757137189
[Diagnostic] imp_factor=1.0, π=(1, 0) feasible, obj=0.9282 (checking tightness next...)
Breaking out because of tightness constraint  9.197095809914734e-06 0.18120192204678265
[Diagnostic] imp_factor=2.0, π=(0, 1) feasible, obj=-0.1358 (checking tightness next...)
[Diagnostic] imp_factor=2.0, π=(1, 0) feasible, obj=-0.1358 (checking tightness next...)
[Diagnostic] imp_factor=1.5, π=(0, 1) feasible, obj=0.6369 (checking tightness next...)
Breaking out because of tightness constraint  0.2818687992071926 0.31627289630080774
[Diagnostic] imp_factor=1.5, π=(1, 0) feasible, obj=0.6369 (checking tightness next...)
Breaking out because of tightness constraint  0.2818714362534392 0.33517694266010706
[Diagnostic] imp_factor=1.75, π=(0, 1) feasible, obj=0.2980 (checking tightness next...)
Breaking out because of tightness c

In [6]:
## Multi tone with one main channel component zeroed
# Detailed analysis with visualization-ready data
U = 2
F = 2

H = np.zeros((F, U, U))

# Create realistic frequency-selective channel
for f in range(F):
    # H[f] = np.eye(U)
    H[f] = np.ones((U,U))
    # # Rayleigh-like fading pattern
    # H[f, 0, 1] = 0.5 * (1 + 0.3 * np.cos(4 * np.pi * f / F))
    # H[f, 1, 0] = 0.3 * (1 + 0.5 * np.sin(2 * np.pi * f / F))
    
H[0, 0, 0] = 0.1

# min_rate = np.array([1.0, 1.0])
min_rate = np.array([0.55, 0.55])

opt_sol, opt_powers, opt_data_rates, best_pi = minPIC_multicarrier_solver(U, H, min_rate, F)

print("\n=== Multi-Carrier Analysis (F=16) ===")
print("Optimal Data Rates:", opt_data_rates)
print("Optimal objective value:", opt_sol)

if opt_powers is not None:
    total_power = sum(opt_powers.values())
    print(f"\nTotal Power: {total_power}")
    
    print("\nPower allocation per frequency:")
    for f in range(F):
        power_f = sum(opt_powers[i, j, f] for i in range(U) for j in range(U))
        print(f"  f={f:2d}: {power_f:.6f}")
    
    print("\nPower allocation per user per frequency:")
    for i in range(U):
        print(f"\nUser {i}:")
        for f in range(F):
            power_i_f = sum(opt_powers[i, j, f] for j in range(U))
            if power_i_f > 1e-6:
                print(f"  f={f:2d}: {power_i_f:.6f}")
    
    # Detailed per-(i,j,f) power distribution
    print("\n=== Detailed Power Allocation per Subcarrier and Link ===")
    for f in range(F):
        print(f"\nSubcarrier {f}:")
        for i in range(U):
            for j in range(U):
                p = opt_powers[(i, j, f)]
                if p > 1e-6:  # Only show non-zero powers
                    print(f"  Rxx[{i},{j},{f}] = {p:.6f}")
    
    # Aggregated views
    print("\n=== Aggregate Power per Transmitter (summed over receivers & freqs) ===")
    for j in range(U):
        total_tx = sum(opt_powers[i, j, f] for i in range(U) for f in range(F))
        print(f"  TX {j}: {total_tx:.6f}")
    
    print("\n=== Aggregate Power per Receiver (summed over transmitters & freqs) ===")
    for i in range(U):
        total_rx = sum(opt_powers[i, j, f] for j in range(U) for f in range(F))
        print(f"  RX {i}: {total_rx:.6f}")
        
achieved_rates, rate_matrix = verify_with_order(U, F, H, opt_powers, min_rate, best_pi)

[Diagnostic] imp_factor=1.0, π=(0, 1) feasible, obj=0.9282 (checking tightness next...)
STATUS is  optimal
values are  0.9281712665285741 inf
HUNGAMA
Breaking out because of tightness constraint  3.0001271779445365e-07 0.274999870381593
MOTAGAMA
[Diagnostic] imp_factor=1.0, π=(1, 0) feasible, obj=0.9282 (checking tightness next...)
STATUS is  optimal
values are  0.9281786453593166 inf
HUNGAMA
Breaking out because of tightness constraint  2.324042417092516e-05 0.2750103327272543
MOTAGAMA
[Diagnostic] imp_factor=2.0, π=(0, 1) feasible, obj=0.0466 (checking tightness next...)
STATUS is  optimal
values are  0.046569318300428986 inf
HUNGAMA
[Diagnostic] imp_factor=2.0, π=(1, 0) feasible, obj=0.0466 (checking tightness next...)
STATUS is  optimal
values are  0.046573270601149463 0.046569318300428986
[Diagnostic] imp_factor=1.5, π=(0, 1) feasible, obj=0.6371 (checking tightness next...)
STATUS is  optimal
values are  0.6370710689252459 inf
HUNGAMA
[Diagnostic] imp_factor=1.5, π=(1, 0) feasibl

/var/folders/wd/7s_rgclx5rlc79rrjnspznh00000gn/T/ipykernel_54741/3622226903.py:164: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  best_data_rates_local = {i: np.sum(b[i, j, f].value for j in range(U) for f in range(F)) for i in range(U)}


[Diagnostic] imp_factor=1.25, π=(1, 0) feasible, obj=0.8550 (checking tightness next...)
STATUS is  optimal
values are  0.8549960612317198 inf
HUNGAMA
Breaking out because of tightness constraint  0.2792654631796659 0.42534694868033357
MOTAGAMA
[Diagnostic] imp_factor=1.375, π=(0, 1) feasible, obj=0.7624 (checking tightness next...)
STATUS is  optimal
values are  0.7623668851875467 inf
HUNGAMA
[Diagnostic] imp_factor=1.375, π=(1, 0) feasible, obj=0.7624 (checking tightness next...)
STATUS is  optimal
values are  0.7624136501050094 0.7623668851875467
[Diagnostic] imp_factor=1.3125, π=(0, 1) feasible, obj=0.8130 (checking tightness next...)
STATUS is  optimal
values are  0.8129716244357703 inf
HUNGAMA
[Diagnostic] imp_factor=1.3125, π=(1, 0) feasible, obj=0.8130 (checking tightness next...)
STATUS is  optimal
values are  0.8129885457694539 0.8129716244357703
[Diagnostic] imp_factor=1.28125, π=(0, 1) feasible, obj=0.8351 (checking tightness next...)
STATUS is  optimal
values are  0.835080

In [95]:
## Multi tone with one main channel component zeroed
# Detailed analysis with visualization-ready data
U = 2
F = 4

H = np.zeros((F, U, U))

# Create realistic frequency-selective channel
for f in range(F):
    # H[f] = np.eye(U)
    H[f] = np.ones((U,U))
    # # Rayleigh-like fading pattern
    # H[f, 0, 1] = 0.5 * (1 + 0.3 * np.cos(4 * np.pi * f / F))
    # H[f, 1, 0] = 0.3 * (1 + 0.5 * np.sin(2 * np.pi * f / F))
    
H[0, 0, 0] = 0.0
H[1, 0, 0] = 0.0
H[2, 0, 0] = 0.0

# min_rate = np.array([1.0, 1.0])
min_rate = np.array([0.55, 0.55])

opt_sol, opt_powers, opt_data_rates, best_pi = minPIC_multicarrier_solver(U, H, min_rate, F)

print("\n=== Multi-Carrier Analysis (F=16) ===")
print("Optimal Data Rates:", opt_data_rates)
print("Optimal objective value:", opt_sol)

if opt_powers is not None:
    total_power = sum(opt_powers.values())
    print(f"\nTotal Power: {total_power}")
    
    print("\nPower allocation per frequency:")
    for f in range(F):
        power_f = sum(opt_powers[i, j, f] for i in range(U) for j in range(U))
        print(f"  f={f:2d}: {power_f:.6f}")
    
    print("\nPower allocation per user per frequency:")
    for i in range(U):
        print(f"\nUser {i}:")
        for f in range(F):
            power_i_f = sum(opt_powers[i, j, f] for j in range(U))
            if power_i_f > 1e-6:
                print(f"  f={f:2d}: {power_i_f:.6f}")
    
    # Detailed per-(i,j,f) power distribution
    print("\n=== Detailed Power Allocation per Subcarrier and Link ===")
    for f in range(F):
        print(f"\nSubcarrier {f}:")
        for i in range(U):
            for j in range(U):
                p = opt_powers[(i, j, f)]
                if p > 1e-6:  # Only show non-zero powers
                    print(f"  Rxx[{i},{j},{f}] = {p:.6f}")
    
    # Aggregated views
    print("\n=== Aggregate Power per Transmitter (summed over receivers & freqs) ===")
    for j in range(U):
        total_tx = sum(opt_powers[i, j, f] for i in range(U) for f in range(F))
        print(f"  TX {j}: {total_tx:.6f}")
    
    print("\n=== Aggregate Power per Receiver (summed over transmitters & freqs) ===")
    for i in range(U):
        total_rx = sum(opt_powers[i, j, f] for j in range(U) for f in range(F))
        print(f"  RX {i}: {total_rx:.6f}")
        
achieved_rates, rate_matrix = verify_with_order(U, F, H, opt_powers, min_rate, best_pi)

[Diagnostic] imp_factor=1.0, π=(0, 1) feasible, obj=0.8400 (checking tightness next...)
STATUS is  optimal
values are  0.839978434601469 inf
HUNGAMA
Breaking out because of tightness constraint  -8.300246599615893e-07 0.13749954693734828
MOTAGAMA
[Diagnostic] imp_factor=1.0, π=(1, 0) feasible, obj=0.8400 (checking tightness next...)
STATUS is  optimal
values are  0.8399785636218561 inf
HUNGAMA
Breaking out because of tightness constraint  1.118534637639643e-06 0.1375001359738531
MOTAGAMA
[Diagnostic] imp_factor=2.0, π=(0, 1) feasible, obj=0.8400 (checking tightness next...)
STATUS is  optimal
values are  0.839978434601469 inf
HUNGAMA
Breaking out because of tightness constraint  -8.300246599615893e-07 0.13749954693734828
MOTAGAMA
[Diagnostic] imp_factor=2.0, π=(1, 0) feasible, obj=0.8400 (checking tightness next...)
STATUS is  optimal
values are  0.8399785636218561 inf
HUNGAMA
Breaking out because of tightness constraint  1.118534637639643e-06 0.1375001359738531
MOTAGAMA
[Diagnostic] i

AttributeError: 'NoneType' object has no attribute 'index'

In [ ]:
## Multi tone with one main channel component zeroed
# Detailed analysis with visualization-ready data
U = 2
F = 2

H = np.zeros((F, U, U))

# Create realistic frequency-selective channel
for f in range(F):
    # H[f] = np.eye(U)
    H[f] = np.ones((U,U))*0.8
    # # Rayleigh-like fading pattern
    # H[f, 0, 1] = 0.5 * (1 + 0.3 * np.cos(4 * np.pi * f / F))
    # H[f, 1, 0] = 0.3 * (1 + 0.5 * np.sin(2 * np.pi * f / F))
    
H[0, 0, 0] = 1.0
H[0, 1, 1] = 1.0
H[0, 2, 2] = 1.0

# min_rate = np.array([1.0, 1.0])
min_rate = np.array([0.5, 0.5, 0.5])

opt_sol, opt_powers, opt_data_rates, best_pi = minPIC_multicarrier_solver(U, H, min_rate, F)

print("\n=== Multi-Carrier Analysis (F=16) ===")
print("Optimal Data Rates:", opt_data_rates)
print("Optimal objective value:", opt_sol)

if opt_powers is not None:
    total_power = sum(opt_powers.values())
    print(f"\nTotal Power: {total_power}")
    
    print("\nPower allocation per frequency:")
    for f in range(F):
        power_f = sum(opt_powers[i, j, f] for i in range(U) for j in range(U))
        print(f"  f={f:2d}: {power_f:.6f}")
    
    print("\nPower allocation per user per frequency:")
    for i in range(U):
        print(f"\nUser {i}:")
        for f in range(F):
            power_i_f = sum(opt_powers[i, j, f] for j in range(U))
            if power_i_f > 1e-6:
                print(f"  f={f:2d}: {power_i_f:.6f}")
    
    # Detailed per-(i,j,f) power distribution
    print("\n=== Detailed Power Allocation per Subcarrier and Link ===")
    for f in range(F):
        print(f"\nSubcarrier {f}:")
        for i in range(U):
            for j in range(U):
                p = opt_powers[(i, j, f)]
                if p > 1e-6:  # Only show non-zero powers
                    print(f"  Rxx[{i},{j},{f}] = {p:.6f}")
    
    # Aggregated views
    print("\n=== Aggregate Power per Transmitter (summed over receivers & freqs) ===")
    for j in range(U):
        total_tx = sum(opt_powers[i, j, f] for i in range(U) for f in range(F))
        print(f"  TX {j}: {total_tx:.6f}")
    
    print("\n=== Aggregate Power per Receiver (summed over transmitters & freqs) ===")
    for i in range(U):
        total_rx = sum(opt_powers[i, j, f] for j in range(U) for f in range(F))
        print(f"  RX {i}: {total_rx:.6f}")
        
achieved_rates, rate_matrix = verify_with_order(U, F, H, opt_powers, min_rate, best_pi)

[Diagnostic] imp_factor=1.0, π=(0, 1, 2) feasible, obj=1.3158 (checking tightness next...)
STATUS is  optimal
values are  1.315788406522312 inf
HUNGAMA
Breaking out because of tightness constraint  7.104606534246276e-07 0.3214176720055189
MOTAGAMA
[Diagnostic] imp_factor=1.0, π=(0, 2, 1) feasible, obj=1.3158 (checking tightness next...)
STATUS is  optimal
values are  1.3157884065224572 inf
HUNGAMA
Breaking out because of tightness constraint  7.104606228469882e-07 0.32141767200570126
MOTAGAMA
[Diagnostic] imp_factor=1.0, π=(1, 0, 2) feasible, obj=1.3158 (checking tightness next...)
STATUS is  optimal
values are  1.3157884065225516 inf
HUNGAMA
Breaking out because of tightness constraint  -1.4398080704642793e-06 0.3214180764022592
MOTAGAMA
[Diagnostic] imp_factor=1.0, π=(1, 2, 0) feasible, obj=1.3158 (checking tightness next...)
STATUS is  optimal
values are  1.315788406522361 inf
HUNGAMA
Breaking out because of tightness constraint  -3.3898731426511114e-08 0.321427460061459
MOTAGAMA
[D

/var/folders/wd/7s_rgclx5rlc79rrjnspznh00000gn/T/ipykernel_54741/3622226903.py:164: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  best_data_rates_local = {i: np.sum(b[i, j, f].value for j in range(U) for f in range(F)) for i in range(U)}


[Diagnostic] imp_factor=32.0, π=(1, 0, 2) feasible, obj=-158.5211 (checking tightness next...)
STATUS is  optimal
values are  -158.5211485262061 -158.52114852649572
[Diagnostic] imp_factor=32.0, π=(1, 2, 0) feasible, obj=-158.5211 (checking tightness next...)
STATUS is  optimal
values are  -158.5211485265906 -158.52114852649572
HUNGAMA
[Diagnostic] imp_factor=32.0, π=(2, 0, 1) feasible, obj=-158.5211 (checking tightness next...)
STATUS is  optimal
values are  -158.52114852616728 -158.5211485265906
[Diagnostic] imp_factor=32.0, π=(2, 1, 0) feasible, obj=-158.5211 (checking tightness next...)
STATUS is  optimal
values are  -158.52114852632639 -158.5211485265906
[Diagnostic] imp_factor=24.0, π=(0, 1, 2) feasible, obj=-104.2795 (checking tightness next...)
STATUS is  optimal
values are  -104.27947865053459 inf
HUNGAMA
[Diagnostic] imp_factor=24.0, π=(0, 2, 1) feasible, obj=-104.2795 (checking tightness next...)
STATUS is  optimal
values are  -104.2794786505802 -104.27947865053459
HUNGAMA
[

In [8]:
## Multi tone with one main channel component zeroed
# Detailed analysis with visualization-ready data
U = 2
F = 16

H = np.zeros((F, U, U))

# Create realistic frequency-selective channel
for f in range(F):
    H[f] = np.eye(U)
    # H[f] = np.ones((U,U))
    # # Rayleigh-like fading pattern
    # H[f, 0, 1] = 0.5 * (1 + 0.3 * np.cos(4 * np.pi * f / F))
    # H[f, 1, 0] = 0.3 * (1 + 0.5 * np.sin(2 * np.pi * f / F))
    
H[0, 0, 1] = 0.9
H[0, 1, 0] = 0.9

# min_rate = np.array([1.0, 1.0])
min_rate = np.array([0.5, 0.5])

opt_sol, opt_powers, opt_data_rates, best_pi = minPIC_multicarrier_solver(U, H, min_rate, F)

print("\n=== Multi-Carrier Analysis (F=16) ===")
print("Optimal Data Rates:", opt_data_rates)
print("Optimal objective value:", opt_sol)

if opt_powers is not None:
    total_power = sum(opt_powers.values())
    print(f"\nTotal Power: {total_power}")
    
    print("\nPower allocation per frequency:")
    for f in range(F):
        power_f = sum(opt_powers[i, j, f] for i in range(U) for j in range(U))
        print(f"  f={f:2d}: {power_f:.6f}")
    
    print("\nPower allocation per user per frequency:")
    for i in range(U):
        print(f"\nUser {i}:")
        for f in range(F):
            power_i_f = sum(opt_powers[i, j, f] for j in range(U))
            if power_i_f > 1e-6:
                print(f"  f={f:2d}: {power_i_f:.6f}")
    
    # Detailed per-(i,j,f) power distribution
    print("\n=== Detailed Power Allocation per Subcarrier and Link ===")
    for f in range(F):
        print(f"\nSubcarrier {f}:")
        for i in range(U):
            for j in range(U):
                p = opt_powers[(i, j, f)]
                if p > 1e-6:  # Only show non-zero powers
                    print(f"  Rxx[{i},{j},{f}] = {p:.6f}")
    
    # Aggregated views
    print("\n=== Aggregate Power per Transmitter (summed over receivers & freqs) ===")
    for j in range(U):
        total_tx = sum(opt_powers[i, j, f] for i in range(U) for f in range(F))
        print(f"  TX {j}: {total_tx:.6f}")
    
    print("\n=== Aggregate Power per Receiver (summed over transmitters & freqs) ===")
    for i in range(U):
        total_rx = sum(opt_powers[i, j, f] for j in range(U) for f in range(F))
        print(f"  RX {i}: {total_rx:.6f}")
        
achieved_rates, rate_matrix = verify_with_order(U, F, H, opt_powers, min_rate, best_pi)

[Diagnostic] imp_factor=1.0, π=(0, 1) feasible, obj=1.0953 (checking tightness next...)
STATUS is  optimal
values are  1.095293873453228 inf
HUNGAMA
Breaking out because of tightness constraint  1.3616096518424867e-06 0.22584866293745182
MOTAGAMA
[Diagnostic] imp_factor=1.0, π=(1, 0) feasible, obj=1.0953 (checking tightness next...)
STATUS is  optimal
values are  1.0952938734534863 inf
HUNGAMA
Breaking out because of tightness constraint  3.2842182016036186e-06 0.2257513219741087
MOTAGAMA
[Diagnostic] imp_factor=2.0, π=(0, 1) feasible, obj=0.7573 (checking tightness next...)
STATUS is  optimal
values are  0.7572592291025897 inf
HUNGAMA


/var/folders/wd/7s_rgclx5rlc79rrjnspznh00000gn/T/ipykernel_54741/3622226903.py:164: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  best_data_rates_local = {i: np.sum(b[i, j, f].value for j in range(U) for f in range(F)) for i in range(U)}


[Diagnostic] imp_factor=2.0, π=(1, 0) feasible, obj=0.7573 (checking tightness next...)
STATUS is  optimal
values are  0.757259229101787 0.7572592291025897
HUNGAMA
[Diagnostic] imp_factor=1.5, π=(0, 1) feasible, obj=1.0517 (checking tightness next...)
STATUS is  optimal
values are  1.051654983117691 inf
HUNGAMA
[Diagnostic] imp_factor=1.5, π=(1, 0) feasible, obj=1.0517 (checking tightness next...)
STATUS is  optimal
values are  1.0516549831160464 1.051654983117691
HUNGAMA
[Diagnostic] imp_factor=1.25, π=(0, 1) feasible, obj=1.0953 (checking tightness next...)
STATUS is  optimal
values are  1.0952973527245382 inf
HUNGAMA
Breaking out because of tightness constraint  -6.360907181312271e-07 0.22549631426606837
MOTAGAMA
[Diagnostic] imp_factor=1.25, π=(1, 0) feasible, obj=1.0953 (checking tightness next...)
STATUS is  optimal
values are  1.0952973527249577 inf
HUNGAMA
Breaking out because of tightness constraint  -5.4593646590902814e-06 0.226105667871865
MOTAGAMA
[Diagnostic] imp_factor=1.

In [9]:
# Test with 3 users and 8 frequency subcarriers
U = 3
F = 16

# Create frequency-selective channel
H = np.zeros((F, U, U))

for f in range(F):
    H[f] = np.eye(U)
    # Add frequency-selective interference pattern
    # H[f, 0, 1] = 0.5 * np.cos(2 * np.pi * f / F)
    # H[f, 0, 2] = 0.001
    # H[f, 1, 0] = 0.001
    # H[f, 1, 2] = 0.3 * np.sin(2 * np.pi * f / F)
    # H[f, 2, 0] = 0.001
    # H[f, 2, 1] = 0.001
    
H[0, :, :] = 0.8
H[0, 0, 0] = H[0, 1, 1] = H[0, 2, 2] = 1.0

min_rate = np.array([0.5, 0.5, 0.5])

opt_sol, opt_powers, opt_data_rates, best_pi = minPIC_multicarrier_solver(U, H, min_rate, F)

print("Optimal decoding order ", best_pi)

print("\n=== Results ===")
print("Optimal Data Rates:", opt_data_rates)
print("Optimal objective value:", opt_sol)

if opt_powers is not None:
    total_power = sum(opt_powers.values())
    print(f"Total Power: {total_power}")
    
    print("\nPower per frequency:")
    for f in range(F):
        power_f = sum(opt_powers[i, j, f] for i in range(U) for j in range(U))
        print(f"  Frequency {f}: {power_f:.4f}")
    
    # Detailed per-(i,j,f) power distribution
    print("\n=== Detailed Power Allocation per Subcarrier and Link ===")
    for f in range(F):
        print(f"\nSubcarrier {f}:")
        for i in range(U):
            for j in range(U):
                p = opt_powers[(i, j, f)]
                print(f"  Rxx[{i},{j},{f}] = {p:.6f}")
    
    # Aggregated views
    print("\n=== Aggregate Power per Transmitter (summed over receivers & freqs) ===")
    for j in range(U):
        total_tx = sum(opt_powers[i, j, f] for i in range(U) for f in range(F))
        print(f"  TX {j}: {total_tx:.6f}")
    
    print("\n=== Aggregate Power per Receiver (summed over transmitters & freqs) ===")
    for i in range(U):
        total_rx = sum(opt_powers[i, j, f] for j in range(U) for f in range(F))
        print(f"  RX {i}: {total_rx:.6f}")
        
        
achieved_rates, rate_matrix = verify_with_order(U, F, H, opt_powers, min_rate, best_pi)

[Diagnostic] imp_factor=1.0, π=(0, 1, 2) feasible, obj=1.3158 (checking tightness next...)
STATUS is  optimal
values are  1.3157885106630154 inf
HUNGAMA
Breaking out because of tightness constraint  -6.792382603069417e-08 0.32142158984284924
MOTAGAMA
[Diagnostic] imp_factor=1.0, π=(0, 2, 1) feasible, obj=1.3158 (checking tightness next...)
STATUS is  optimal
values are  1.3157885106551854 inf
HUNGAMA
Breaking out because of tightness constraint  -6.792384644990566e-08 0.32142158982569413
MOTAGAMA
[Diagnostic] imp_factor=1.0, π=(1, 0, 2) feasible, obj=1.3158 (checking tightness next...)
STATUS is  optimal
values are  1.3157885106628089 inf
HUNGAMA
Breaking out because of tightness constraint  -7.298528677314182e-08 0.32142242052275327
MOTAGAMA
[Diagnostic] imp_factor=1.0, π=(1, 2, 0) feasible, obj=1.3158 (checking tightness next...)
STATUS is  optimal
values are  1.3157885106417846 inf
HUNGAMA
Breaking out because of tightness constraint  -1.4462661034734238e-07 0.32141891748238277
MOTA

/var/folders/wd/7s_rgclx5rlc79rrjnspznh00000gn/T/ipykernel_54741/3622226903.py:164: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  best_data_rates_local = {i: np.sum(b[i, j, f].value for j in range(U) for f in range(F)) for i in range(U)}


[Diagnostic] imp_factor=2.0, π=(0, 2, 1) feasible, obj=0.4257 (checking tightness next...)
STATUS is  optimal
values are  0.425710314262699 0.42571031426575034
HUNGAMA
[Diagnostic] imp_factor=2.0, π=(1, 0, 2) feasible, obj=0.4257 (checking tightness next...)
STATUS is  optimal
values are  0.425710314252572 0.425710314262699
HUNGAMA
[Diagnostic] imp_factor=2.0, π=(1, 2, 0) feasible, obj=0.4257 (checking tightness next...)
STATUS is  optimal
values are  0.42571031425698624 0.425710314252572
[Diagnostic] imp_factor=2.0, π=(2, 0, 1) feasible, obj=0.4257 (checking tightness next...)
STATUS is  optimal
values are  0.4257103142585894 0.425710314252572
[Diagnostic] imp_factor=2.0, π=(2, 1, 0) feasible, obj=0.4257 (checking tightness next...)
STATUS is  optimal
values are  0.42571031425365646 0.425710314252572
[Diagnostic] imp_factor=1.5, π=(0, 1, 2) feasible, obj=1.1680 (checking tightness next...)
STATUS is  optimal
values are  1.1680260141967223 inf
HUNGAMA
[Diagnostic] imp_factor=1.5, π=(0,